# Fine-tuning QLoRA del redactor del agente (Ruta 1)

**Objetivo:** ajustar `Qwen2.5-3B-Instruct` (3 000 millones de parámetros) con QLoRA para que redacte respuestas con
el formato del agente: **Respuesta / Explicación / Fuentes**, citando los fragmentos recuperados y usando el
resultado de la herramienta sin recalcular. El fine-tuning aporta la **forma**; el **contenido** viene del RAG.

**Entorno:** Google Colab con GPU T4 gratuita (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*) o Kaggle (GPU T4 x2).
Tiempo aproximado: 20–40 minutos en total.

**Salidas (evidencia del entrenamiento):**
- `evidencia/curva_perdida.png` y `evidencia/historial_entrenamiento.csv`
- `evidencia/configuracion.json` (hiperparámetros, versiones, GPU)
- `evidencia/muestras_validacion.md` (respuestas del modelo ajustado sobre el conjunto de validación)
- `lora_adapter/` (adaptador LoRA) y `arqui-qwen2.5-3b-ft.Q4_K_M.gguf` (modelo fusionado y cuantizado para CPU)

In [ ]:
%%capture
!pip install unsloth

## 1. Datos
Se clona el repositorio para usar exactamente el dataset versionado (`data/finetune/`). Cambie la URL por la de su repositorio.

In [ ]:
REPO = "https://github.com/SU_USUARIO/agente-arquitectura.git"   # <-- cambiar
!rm -rf agente-arquitectura && git clone -q $REPO
!wc -l agente-arquitectura/data/finetune/*.jsonl

## 2. Modelo base cuantizado en 4 bits

In [ ]:
import torch, json, platform
from unsloth import FastLanguageModel

MODELO_BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO_BASE, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
print(torch.cuda.get_device_name(0))

## 3. Adaptador LoRA
Solo se entrenan matrices de bajo rango (r = 16) sobre las proyecciones de atención y del MLP; los pesos base quedan congelados.

In [ ]:
LORA = dict(r=16, lora_alpha=16, lora_dropout=0.0,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = FastLanguageModel.get_peft_model(model, **LORA, bias="none",
                                         use_gradient_checkpointing="unsloth", random_state=2026)
model.print_trainable_parameters()

## 4. Formato de chat
Cada ejemplo ya trae `messages` (sistema, usuario con fragmentos, asistente). Se aplica la plantilla ChatML de Qwen.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": "agente-arquitectura/data/finetune/train.jsonl",
    "validation": "agente-arquitectura/data/finetune/validacion.jsonl"})

def a_texto(lote):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in lote["messages"]]}

ds = ds.map(a_texto, batched=True)
largos = [len(tokenizer(t).input_ids) for t in ds["train"]["text"]]
print(ds, "\nTokens por ejemplo: máx", max(largos), "promedio", sum(largos) // len(largos))

## 5. Entrenamiento
Se calcula la pérdida **solo sobre la respuesta del asistente** (`train_on_responses_only`): el modelo aprende a redactar, no a repetir los fragmentos.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

HIPER = dict(per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=3,
             learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear", weight_decay=0.01,
             optim="adamw_8bit", seed=2026)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ, logging_steps=2,
                   eval_strategy="steps", eval_steps=6, output_dir="outputs", report_to="none", **HIPER))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

## 6. Evidencia del entrenamiento

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt
os.makedirs("evidencia", exist_ok=True)
hist = pd.DataFrame(trainer.state.log_history)
hist.to_csv("evidencia/historial_entrenamiento.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
t = hist.dropna(subset=["loss"]); v = hist.dropna(subset=["eval_loss"])
ax.plot(t["step"], t["loss"], label="entrenamiento")
ax.plot(v["step"], v["eval_loss"], marker="o", label="validación")
ax.set_xlabel("paso"); ax.set_ylabel("pérdida"); ax.set_title("QLoRA Qwen2.5-3B — pérdida"); ax.legend()
fig.savefig("evidencia/curva_perdida.png", dpi=150, bbox_inches="tight"); plt.show()

import transformers, trl, peft
json.dump({"modelo_base": MODELO_BASE, "max_seq_length": MAX_SEQ, "lora": LORA, "hiperparametros": HIPER,
           "ejemplos_train": len(ds["train"]), "ejemplos_validacion": len(ds["validation"]),
           "gpu": torch.cuda.get_device_name(0), "segundos_entrenamiento": stats.metrics["train_runtime"],
           "perdida_final": stats.metrics["train_loss"],
           "versiones": {"torch": torch.__version__, "transformers": transformers.__version__,
                         "trl": trl.__version__, "peft": peft.__version__}},
          open("evidencia/configuracion.json", "w"), indent=2, ensure_ascii=False)

## 7. Prueba rápida sobre el conjunto de validación

In [ ]:
FastLanguageModel.for_inference(model)
salidas = []
for fila in ds["validation"]:
    prompt = tokenizer.apply_chat_template(fila["messages"][:-1], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    gen = model.generate(**ids, max_new_tokens=500, temperature=0.2, do_sample=True)
    texto = tokenizer.decode(gen[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    salidas.append(f"### {fila['id']}\n\n**Esperado:**\n\n{fila['messages'][-1]['content']}\n\n**Modelo ajustado:**\n\n{texto}\n")
open("evidencia/muestras_validacion.md", "w").write("\n".join(salidas))
print(salidas[0])

## 8. Guardar adaptador y exportar a GGUF (CPU)
Se fusiona el adaptador con el modelo base y se cuantiza a `Q4_K_M` (~2 GB) para servirlo con Ollama en el VPS sin GPU.

In [ ]:
model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
import glob, shutil
origen = glob.glob("gguf*/*Q4_K_M*.gguf") + glob.glob("*Q4_K_M*.gguf")
print(origen)
shutil.move(origen[0], "arqui-qwen2.5-3b-ft.Q4_K_M.gguf")
!ls -lh arqui-qwen2.5-3b-ft.Q4_K_M.gguf

## 9. Descargar
Guarde en Google Drive (más rápido para archivos grandes) o descargue directamente. Luego copie el `.gguf` al VPS en `deploy/ollama/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/agente-arquitectura
!cp -r evidencia lora_adapter arqui-qwen2.5-3b-ft.Q4_K_M.gguf /content/drive/MyDrive/agente-arquitectura/
print("Copiado a Drive: MyDrive/agente-arquitectura/")